# 05 — Weak concepts: grouping missed questions

"You got 12 wrong" is not actionable; "you are weak on *mobile phones and the safety of homeless youth*" is.
This notebook picks the clustering threshold for `ml_pipeline/weak_concepts.py` on the dev mistake bank
(every question the dev account currently owes, dumped from the platform to `outputs/owed_questions_dev.json`
— content from the user's own papers, so not committed).

Method: TF-IDF over words (1–2-grams) **and** character n-grams (3–5, `char_wb`) — no language model, so it
works for Vietnamese too — then average-linkage agglomerative clustering on cosine distance with a distance
threshold. Text per question = question + correct answer + source excerpt (when the question has one).

In [1]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
from ml_pipeline.weak_concepts import cluster_questions, top_terms, DISTANCE_THRESHOLD

rows = json.load(open(Path.cwd().resolve().parents[0] / 'outputs' / 'owed_questions_dev.json'))
texts = [f"{r['question']} {r['answer']} {r['excerpt']}" for r in rows]
print(len(rows), 'owed questions;', sum(1 for r in rows if r['excerpt']), 'with a source excerpt')

49 owed questions; 4 with a source excerpt


## Threshold sweep

What matters is not a score but whether the clusters read as one concept each, and how much of the bank they cover.

In [2]:
import pandas as pd
sweep = []
for th in (0.6, 0.7, 0.75, 0.8, 0.82, 0.85, 0.9):
    cl = cluster_questions(texts, threshold=th)
    sweep.append({'threshold': th, 'clusters': len(cl), 'covered': sum(len(c.indices) for c in cl),
                  'largest': max((len(c.indices) for c in cl), default=0),
                  'sizes': [len(c.indices) for c in cl]})
pd.DataFrame(sweep)

,threshold,clusters,covered,largest,sizes
0,0.60,8,18,3,"[3, 3, 2, 2, 2, 2, 2, 2]"
1,0.70,10,26,4,"[4, 3, 3, 3, 3, 2, 2, 2, 2, 2]"
2,0.75,13,32,4,"[4, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2]"
3,0.80,14,35,4,"[4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2]"
4,0.82,15,37,4,"[4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]"
5,0.85,14,41,9,"[9, 4, 4, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2]"
6,0.90,10,45,11,"[11, 7, 6, 5, 5, 3, 2, 2, 2, 2]"


In [3]:
def show(th):
    for c in cluster_questions(texts, threshold=th):
        print(f"[{len(c.indices)}] {', '.join(top_terms([r['question'] for r in rows], c.indices))}")
        for i in c.indices:
            print('     ', rows[i]['question'][:95])
show(DISTANCE_THRESHOLD)

[9] safety, phones, mobile
      According to the authors, what role do mobile phones play in the personal safety of homeless yo
      What role do mobile phones play in the personal safety of homeless young people?
      What does a mobile phone serve as for homeless young people?
      What is one way that phone theft affects the safety benefit of mobile phones for homeless youth
      Why is the safety benefit of phones undermined?
      What is CityHelp's most prominent interaction, according to the paper's demonstration?
      What is the primary safety function demonstrated by the paper?
      How do mobile phones affect the personal safety of homeless young people?
      What is a primary safety function of mobile phones, according to the paper?
[4] pages analyzed, analyzed total, national policy
      What was the final sample size of national AI policy strategies after screening?
      How many pages were analyzed in total by the author?
      What was the final sample size of

## Decision

**0.85.** 41/49 questions grouped into 14 clusters. The largest (9) is a single real concept — mobile phones
and the personal safety of homeless young people — with the safety-function questions correctly pulled in.
At 0.8 that concept is split three ways and Vietnamese pairs sharing only the concept's two-syllable name
(*quang hợp*) fall apart; at 0.9 unrelated CityHelp questions fuse. The 8 leftovers are singletons: missed
once, not yet a pattern.

Labels here come from `top_terms` (the TF-IDF fallback). In the app an LLM names each cluster from its
questions when a provider is available; `top_terms` is what a user sees without one.

In [4]:
out = Path.cwd().resolve().parents[0] / 'reports' / 'weak_concepts_metrics.json'
chosen = next(s for s in sweep if s['threshold'] == DISTANCE_THRESHOLD)
json.dump({'threshold': DISTANCE_THRESHOLD, 'questions': len(rows), **{k: chosen[k] for k in ('clusters', 'covered', 'largest', 'sizes')},
           'sweep': sweep}, open(out, 'w'), indent=2)
print('wrote', out.name)

wrote weak_concepts_metrics.json
